# DreamBooth Concept Fine-Tuning for Rumah Gadang

Notebook pembanding ini memakai pipeline DreamBooth-style fine-tuning pada Stable Diffusion 1.5. Berbeda dari LoRA concept notebook, baseline ini mengubah bobot UNet secara langsung pada trainable scope tertentu, lalu mengevaluasi kemampuan img2img pada file input evaluasi yang sama dengan LoRA.

Split Gadang train/val/test dibuat dengan seed dan aturan yang sama seperti notebook LoRA concept sehingga data pembanding konsisten.

## 1. Setup, Config, and Logging

In [ ]:
from __future__ import annotations

import os
import sys
import re
import json
import random
import itertools
import shutil
import pickle
from pathlib import Path
from datetime import datetime


def env_int(name: str, default: int) -> int:
    value = os.environ.get(name, "").strip()
    return int(value) if value else default


def env_float(name: str, default: float) -> float:
    value = os.environ.get(name, "").strip()
    return float(value) if value else default


def env_str(name: str, default: str) -> str:
    value = os.environ.get(name, "").strip()
    return value if value else default


def slugify(value: str) -> str:
    value = value.replace(".", "p").replace("-", "m")
    return re.sub(r"[^A-Za-z0-9_]+", "_", value).strip("_")


PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent
REQUESTED_DATA_ROOT = PROJECT_ROOT / "data" / "traditional_houses"
FALLBACK_DATA_ROOT = PROJECT_ROOT / "traditional_houses"
DATA_ROOT = REQUESTED_DATA_ROOT if REQUESTED_DATA_ROOT.exists() else FALLBACK_DATA_ROOT

CONCEPT_TOKEN = env_str("GADANG_CONCEPT_TOKEN", "gadang")
TARGET_CLASS = "Gadang"
HOUSE_CLASSES = ["Gadang", "Joglo", "Honai", "Panjang", "Tongkonan"]
CLASS_ALIASES = {name.lower(): name for name in HOUSE_CLASSES}
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

RESOLUTION = env_int("GADANG_RESOLUTION", 512)
MAX_CONCEPT_IMAGES = env_int("GADANG_CONCEPT_IMAGES", 10)
MAX_TRAIN_STEPS = env_int("GADANG_MAX_TRAIN_STEPS", 1000)
CHECKPOINT_EVERY = env_int("GADANG_CHECKPOINT_EVERY", 100)
LEARNING_RATE = env_float("GADANG_LEARNING_RATE", 1e-5)
DREAMBOOTH_SCOPE_LABEL = env_str("GADANG_DREAMBOOTH_SCOPE", env_str("GADANG_LORA_RANK", "16"))
TRAIN_BATCH_SIZE = env_int("GADANG_TRAIN_BATCH_SIZE", 1)
GRADIENT_ACCUMULATION_STEPS = env_int("GADANG_GRADIENT_ACCUMULATION_STEPS", 4)
MIXED_PRECISION = env_str("GADANG_MIXED_PRECISION", "fp16")
VAL_IMAGE_COUNT = env_int("GADANG_VAL_IMAGES", 10)
TEST_IMAGE_COUNT = env_int("GADANG_TEST_IMAGES", 10)
RUN_EVALUATION = env_int("GADANG_RUN_EVALUATION", 1) == 1
RUN_METRICS = env_int("GADANG_RUN_METRICS", 1) == 1
EVAL_IMAGES_PER_CLASS = env_int("GADANG_EVAL_IMAGES_PER_CLASS", 2)
EVAL_PROMPT_COUNT = env_int("GADANG_EVAL_PROMPT_COUNT", 3)
EVAL_NUM_INFERENCE_STEPS = env_int("GADANG_EVAL_NUM_INFERENCE_STEPS", 30)
EVAL_GUIDANCE_SCALE = env_float("GADANG_EVAL_GUIDANCE_SCALE", 7.5)
EVAL_IMAGE_GUIDANCE_SCALE = env_float("GADANG_EVAL_IMAGE_GUIDANCE_SCALE", 1.5)
EVAL_STRENGTH = env_float("GADANG_EVAL_STRENGTH", 0.75)
RUN_STAMP = env_str("GADANG_RUN_STAMP", datetime.now().strftime("%Y%m%d_%H%M%S"))
SEED = env_int("GADANG_SEED", 100)

random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
os.environ.setdefault("TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD", "1")

lr_slug = slugify(f"{LEARNING_RATE:.0e}") if LEARNING_RATE < 0.001 else slugify(str(LEARNING_RATE))
DEFAULT_RUN_NAME = f"img{MAX_CONCEPT_IMAGES}_lr{lr_slug}_scope{slugify(str(DREAMBOOTH_SCOPE_LABEL))}_steps{MAX_TRAIN_STEPS}"
RUN_NAME = slugify(env_str("GADANG_RUN_NAME", DEFAULT_RUN_NAME))

CONCEPT_BASE_MODEL_ID = "local:v1-5-pruned-emaonly.ckpt"
LOCAL_SD_CKPT_PATH = Path(env_str("GADANG_STABLE_DIFFUSION_CKPT", str(PROJECT_ROOT / "models" / "v1-5-pruned-emaonly.ckpt")))
OUTPUT_DIR = PROJECT_ROOT / "outputs" / RUN_NAME
CONCEPT_DATA_ROOT = OUTPUT_DIR / "concept_dataset"
CONCEPT_IMAGES_DIR = CONCEPT_DATA_ROOT / "images"
CONCEPT_METADATA_PATH = CONCEPT_DATA_ROOT / "metadata.jsonl"
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
EVALUATION_DIR = OUTPUT_DIR / "evaluation"
METRICS_DIR = OUTPUT_DIR / "metrics"
LOG_DIR = OUTPUT_DIR / "logs"
NOTEBOOK_OUTPUT_DIR = OUTPUT_DIR / "notebook"
RESULTS_DIR = EVALUATION_DIR
RUN_CONFIG_PATH = OUTPUT_DIR / "run_config.json"

for directory in [OUTPUT_DIR, CONCEPT_IMAGES_DIR, CHECKPOINT_DIR, EVALUATION_DIR, METRICS_DIR, LOG_DIR, NOTEBOOK_OUTPUT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

ROOT_LOG_PATH = PROJECT_ROOT / "output.log"
RUN_LOG_PATH = LOG_DIR / "run_output.log"
NOTEBOOK_WRITES_ROOT_LOG = env_int("GADANG_NOTEBOOK_WRITES_ROOT_LOG", 0) == 1

class Tee:
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)
            stream.flush()

    def flush(self):
        for stream in self.streams:
            stream.flush()

_run_log_file = RUN_LOG_PATH.open("w", encoding="utf-8")
if NOTEBOOK_WRITES_ROOT_LOG:
    _root_log_file = ROOT_LOG_PATH.open("w", encoding="utf-8")
    sys.stdout = Tee(sys.__stdout__, _root_log_file, _run_log_file)
    sys.stderr = Tee(sys.__stderr__, _root_log_file, _run_log_file)
else:
    sys.stdout = Tee(sys.__stdout__, _run_log_file)
    sys.stderr = Tee(sys.__stderr__, _run_log_file)

RUN_CONFIG = {
    "method": "dreambooth",
    "run_name": RUN_NAME,
    "run_stamp": RUN_STAMP,
    "concept_token": CONCEPT_TOKEN,
    "max_concept_images": MAX_CONCEPT_IMAGES,
    "max_train_steps": MAX_TRAIN_STEPS,
    "checkpoint_every": CHECKPOINT_EVERY,
    "learning_rate": LEARNING_RATE,
    "dreambooth_scope_label": DREAMBOOTH_SCOPE_LABEL,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "mixed_precision": MIXED_PRECISION,
    "resolution": RESOLUTION,
    "seed": SEED,
    "run_evaluation": RUN_EVALUATION,
    "run_metrics": RUN_METRICS,
    "eval_images_per_class": EVAL_IMAGES_PER_CLASS,
    "eval_strength": EVAL_STRENGTH,
    "val_image_count": VAL_IMAGE_COUNT,
    "test_image_count": TEST_IMAGE_COUNT,
    "concept_base_model": CONCEPT_BASE_MODEL_ID,
    "local_stable_diffusion_checkpoint": str(LOCAL_SD_CKPT_PATH),
    "loaded_local_stable_diffusion_checkpoint": bool(LOCAL_SD_CKPT_PATH.exists()),
}
RUN_CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")

def log_event(message: str) -> None:
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}", flush=True)


def log_section(title: str) -> None:
    log_event("=" * 72)
    log_event(title)
    log_event("=" * 72)


log_section("Notebook logging initialized")
log_event(f"Root output log: {ROOT_LOG_PATH} (handled by runner; notebook direct write={NOTEBOOK_WRITES_ROOT_LOG})")
log_event(f"Run output log: {RUN_LOG_PATH}")
log_section("Run configuration")
log_event(f"Project root: {PROJECT_ROOT}")
log_event(f"Dataset root: {DATA_ROOT}")
log_event(f"Run name: {RUN_NAME}")
log_event(f"Run stamp: {RUN_STAMP}")
log_event("Run config JSON:")
print(json.dumps(RUN_CONFIG, indent=2), flush=True)
assert DATA_ROOT.exists(), f"Dataset directory was not found: {DATA_ROOT}"


## 2. Dependency Verification and Dataset Loading

In [ ]:
import torch
import torchvision
import diffusers
import transformers
import accelerate
import PIL
import matplotlib
import pandas as pd
from PIL import Image, ImageOps

print("torch:", torch.__version__)
print("diffusers:", diffusers.__version__)
print("transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


def canonical_class_name(path: Path) -> str:
    return CLASS_ALIASES.get(path.name.lower(), path.name.capitalize())


def discover_images(data_root: Path) -> pd.DataFrame:
    rows = []
    for class_dir in sorted([p for p in data_root.iterdir() if p.is_dir()]):
        class_name = canonical_class_name(class_dir)
        if class_name not in HOUSE_CLASSES:
            continue
        for image_path in sorted(class_dir.rglob("*")):
            if image_path.suffix.lower() in IMAGE_EXTENSIONS:
                rows.append({"class_name": class_name, "image_path": str(image_path)})
    return pd.DataFrame(rows)

images_df = discover_images(DATA_ROOT)
print(images_df.groupby("class_name").size())
assert (images_df["class_name"] == TARGET_CLASS).any(), "No Gadang images found."

## 3. Deterministic Train/Val/Test Split

In [ ]:
log_section("Dependency verification and dataset loading")
from tqdm.auto import tqdm

CAPTION_TEMPLATES = [
    "a photo of gadang, traditional Minangkabau house from West Sumatra",
    "a realistic photo of gadang with iconic curved gonjong roof",
    "traditional wooden gadang house architecture from West Sumatra",
    "front view of gadang, Indonesian traditional house with curved roof",
    "a detailed architectural photo of gadang, Rumah Gadang Minangkabau house",
]


def resize_square(image: Image.Image, size: int = RESOLUTION) -> Image.Image:
    image = ImageOps.exif_transpose(image).convert("RGB")
    width, height = image.size
    crop_size = min(width, height)
    left = (width - crop_size) // 2
    top = (height - crop_size) // 2
    image = image.crop((left, top, left + crop_size, top + crop_size))
    return image.resize((size, size), Image.Resampling.LANCZOS)


gadang_df = images_df[images_df["class_name"] == TARGET_CLASS].sample(frac=1.0, random_state=SEED).reset_index(drop=True)
needed = MAX_CONCEPT_IMAGES + VAL_IMAGE_COUNT + TEST_IMAGE_COUNT
assert len(gadang_df) >= needed, f"Need at least {needed} Gadang images."
concept_df = gadang_df.iloc[:MAX_CONCEPT_IMAGES].copy().reset_index(drop=True)
val_df = gadang_df.iloc[MAX_CONCEPT_IMAGES:MAX_CONCEPT_IMAGES + VAL_IMAGE_COUNT].copy().reset_index(drop=True)
test_df = gadang_df.iloc[MAX_CONCEPT_IMAGES + VAL_IMAGE_COUNT:needed].copy().reset_index(drop=True)

shutil.rmtree(CONCEPT_IMAGES_DIR, ignore_errors=True)
CONCEPT_IMAGES_DIR.mkdir(parents=True, exist_ok=True)
records = []
for idx, row in tqdm(concept_df.iterrows(), total=len(concept_df), desc="Preparing DreamBooth images"):
    source_path = Path(row["image_path"])
    output_path = CONCEPT_IMAGES_DIR / f"gadang_train_{idx:05d}.png"
    resize_square(Image.open(source_path), RESOLUTION).save(output_path)
    records.append({
        "image": str(output_path),
        "caption": CAPTION_TEMPLATES[idx % len(CAPTION_TEMPLATES)],
        "class_name": TARGET_CLASS,
        "source_image": str(source_path),
        "split": "train",
        "run_name": RUN_NAME,
    })

with CONCEPT_METADATA_PATH.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record) + "\n")

SPLIT_MANIFEST_PATH = CONCEPT_DATA_ROOT / "split_manifest.json"
SPLIT_MANIFEST_PATH.write_text(json.dumps({
    "run_name": RUN_NAME,
    "seed": SEED,
    "train": concept_df["image_path"].tolist(),
    "val": val_df["image_path"].tolist(),
    "test": test_df["image_path"].tolist(),
}, indent=2), encoding="utf-8")
print(f"Train images: {len(concept_df)} | Val: {len(val_df)} | Test: {len(test_df)}")
print(f"Saved split manifest: {SPLIT_MANIFEST_PATH}")

## 4. DreamBooth-Style Fine-Tuning

In [ ]:
log_section("Dataset discovery")
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from diffusers import StableDiffusionPipeline, DDPMScheduler
from accelerate import Accelerator
import torch.nn.functional as F

image_transform = transforms.Compose([
    transforms.Resize((RESOLUTION, RESOLUTION), interpolation=transforms.InterpolationMode.BILINEAR),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),
])

class DreamBoothConceptDataset(Dataset):
    def __init__(self, metadata_path: Path):
        self.records = [json.loads(line) for line in metadata_path.read_text().splitlines() if line.strip()]

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        image = Image.open(record["image"]).convert("RGB")
        return {"pixel_values": image_transform(image), "caption": record["caption"]}


def collate_fn(batch):
    return {"pixel_values": torch.stack([x["pixel_values"] for x in batch]), "caption": [x["caption"] for x in batch]}


def configure_trainable_unet(unet, scope_label: str):
    scope = str(scope_label).lower()
    if scope in {"4", "attn_proj", "small"}:
        mode = "attention_projection_only"
        keywords = ["to_q", "to_k", "to_v", "to_out"]
        for name, param in unet.named_parameters():
            param.requires_grad = any(key in name for key in keywords)
    elif scope in {"8", "attention", "medium"}:
        mode = "all_attention_blocks"
        for name, param in unet.named_parameters():
            param.requires_grad = "attn" in name
    else:
        mode = "full_unet"
        for param in unet.parameters():
            param.requires_grad = True
    return mode


def count_trainable_parameters(model):
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return trainable, total



def enable_trusted_legacy_ckpt_loading():
    if getattr(torch.load, "_gadang_trusted_ckpt_patch", False):
        return
    original_torch_load = torch.load

    def trusted_torch_load(*args, **kwargs):
        kwargs["weights_only"] = False
        return original_torch_load(*args, **kwargs)

    trusted_torch_load._gadang_trusted_ckpt_patch = True
    torch.load = trusted_torch_load
    log_event("Enabled trusted legacy .ckpt loading with torch.load(weights_only=False) for local checkpoints")


def load_stable_diffusion_pipeline(pipeline_cls):
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    assert LOCAL_SD_CKPT_PATH.exists(), f"Stable Diffusion checkpoint not found: {LOCAL_SD_CKPT_PATH}"
    log_event(f"Loading local Stable Diffusion checkpoint: {LOCAL_SD_CKPT_PATH}")
    enable_trusted_legacy_ckpt_loading()
    return pipeline_cls.from_single_file(
        str(LOCAL_SD_CKPT_PATH),
        torch_dtype=dtype,
        safety_checker=None,
    )

train_dataset = DreamBoothConceptDataset(CONCEPT_METADATA_PATH)
train_dataloader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True, collate_fn=collate_fn, num_workers=0, pin_memory=True)

accelerator = Accelerator(gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS, mixed_precision=MIXED_PRECISION if torch.cuda.is_available() else "no")
pipe = load_stable_diffusion_pipeline(StableDiffusionPipeline)
noise_scheduler = DDPMScheduler.from_config(pipe.scheduler.config)
vae, text_encoder, tokenizer, unet = pipe.vae, pipe.text_encoder, pipe.tokenizer, pipe.unet
vae.requires_grad_(False)
text_encoder.requires_grad_(False)
trainable_mode = configure_trainable_unet(unet, DREAMBOOTH_SCOPE_LABEL)
# Trainable DreamBooth parameters must stay in fp32. If they remain fp16,
# GradScaler cannot unscale gradients and raises: Attempting to unscale FP16 gradients.
unet.to(dtype=torch.float32)
trainable_params, total_params = count_trainable_parameters(unet)
print(f"DreamBooth trainable mode: {trainable_mode}")
print(f"Trainable parameters: {trainable_params:,} / {total_params:,}")
print("UNet dtype for trainable DreamBooth parameters: float32")

optimizer = torch.optim.AdamW([p for p in unet.parameters() if p.requires_grad], lr=LEARNING_RATE)
unet, optimizer, train_dataloader = accelerator.prepare(unet, optimizer, train_dataloader)
vae.to(accelerator.device)
text_encoder.to(accelerator.device)
vae.eval(); text_encoder.eval()
weight_dtype = torch.float16 if accelerator.mixed_precision == "fp16" else torch.float32
vae.to(dtype=weight_dtype); text_encoder.to(dtype=weight_dtype)

log_section("Training start")
log_event(f"Optimizer-step budget: {MAX_TRAIN_STEPS}")
log_event(f"Train batch size: {TRAIN_BATCH_SIZE}; gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
log_event(f"Learning rate: {LEARNING_RATE}")
log_event(f"Checkpoint interval: every {CHECKPOINT_EVERY} optimizer steps")
checkpoint_records = []

def save_dreambooth_checkpoint(step: int, label: str):
    accelerator.wait_for_everyone()
    if accelerator.is_main_process:
        checkpoint_root = CHECKPOINT_DIR / label
        checkpoint_root.mkdir(parents=True, exist_ok=True)
        current_unet = accelerator.unwrap_model(unet)
        pipe.unet = current_unet
        pipe.save_pretrained(checkpoint_root / "pipeline")
        current_unet.save_pretrained(checkpoint_root / "unet")
        record = {"step": int(step), "label": label, "pipeline_path": str(checkpoint_root / "pipeline"), "unet_path": str(checkpoint_root / "unet")}
        checkpoint_records.append(record)
        log_event(f"Saved DreamBooth checkpoint {label} at step {step}: {checkpoint_root}")

loss_history = []
global_step = 0
micro_step = 0
batch_iterator = itertools.cycle(train_dataloader)
unet.train()
progress_bar = tqdm(total=MAX_TRAIN_STEPS, desc=f"{RUN_NAME} optimizer steps", disable=not accelerator.is_local_main_process)
while global_step < MAX_TRAIN_STEPS:
    batch = next(batch_iterator)
    micro_step += 1
    with accelerator.accumulate(unet):
        pixel_values = batch["pixel_values"].to(accelerator.device, dtype=weight_dtype)
        with torch.no_grad():
            latents = vae.encode(pixel_values).latent_dist.sample() * vae.config.scaling_factor
            input_ids = tokenizer(batch["caption"], padding="max_length", truncation=True, max_length=tokenizer.model_max_length, return_tensors="pt").input_ids.to(accelerator.device)
            encoder_hidden_states = text_encoder(input_ids)[0]
        noise = torch.randn_like(latents)
        timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (latents.shape[0],), device=latents.device).long()
        noisy_latents = noise_scheduler.add_noise(latents, noise, timesteps)
        model_pred = unet(noisy_latents, timesteps, encoder_hidden_states).sample
        loss = F.mse_loss(model_pred.float(), noise.float(), reduction="mean")
        accelerator.backward(loss)
        if accelerator.sync_gradients:
            accelerator.clip_grad_norm_(unet.parameters(), 1.0)
        optimizer.step(); optimizer.zero_grad(set_to_none=True)
    if accelerator.sync_gradients:
        global_step += 1
        loss_value = accelerator.gather(loss.detach()).mean().item()
        loss_history.append({"step": global_step, "micro_step": micro_step, "loss": loss_value})
        progress_bar.update(1); progress_bar.set_postfix(loss=f"{loss_value:.4f}")
        if CHECKPOINT_EVERY > 0 and global_step % CHECKPOINT_EVERY == 0:
            save_dreambooth_checkpoint(global_step, f"step_{global_step:04d}")
progress_bar.close()
accelerator.wait_for_everyone()

unwrapped_unet = accelerator.unwrap_model(unet)
pipe.unet = unwrapped_unet
if accelerator.is_main_process:
    final_root = CHECKPOINT_DIR / "final"
    final_root.mkdir(parents=True, exist_ok=True)
    pipe.save_pretrained(final_root / "pipeline")
    unwrapped_unet.save_pretrained(final_root / "unet")
    pipe.save_pretrained(CHECKPOINT_DIR / "pipeline")
    unwrapped_unet.save_pretrained(CHECKPOINT_DIR / "unet")
    if not checkpoint_records or checkpoint_records[-1]["step"] != int(global_step):
        checkpoint_records.append({"step": int(global_step), "label": "final", "pipeline_path": str(final_root / "pipeline"), "unet_path": str(final_root / "unet")})
    (CHECKPOINT_DIR / "checkpoints_manifest.json").write_text(json.dumps({"run_name": RUN_NAME, "checkpoints": checkpoint_records}, indent=2), encoding="utf-8")
pd.DataFrame(loss_history).to_csv(METRICS_DIR / "training_loss.csv", index=False)
RUN_CONFIG.update({"trainable_mode": trainable_mode, "trainable_params": int(trainable_params), "total_params": int(total_params), "actual_optimizer_steps": int(global_step), "actual_micro_steps": int(micro_step), "checkpoint_every": int(CHECKPOINT_EVERY), "checkpoint_count": len(checkpoint_records)})
RUN_CONFIG_PATH.write_text(json.dumps(RUN_CONFIG, indent=2), encoding="utf-8")
log_section("Training finished and artifacts saved")
print(f"Saved DreamBooth pipeline to: {CHECKPOINT_DIR / 'pipeline'}")

## 5. Text-to-Image Evaluation

In [ ]:
from diffusers import StableDiffusionImg2ImgPipeline

EVAL_CLASSES = [class_name for class_name in HOUSE_CLASSES if class_name != TARGET_CLASS]
TEST_PROMPTS = [
    "replace the house with a realistic gadang traditional house from West Sumatra",
    "transform this scene into a gadang house with iconic curved gonjong roof",
    "make the main building look like gadang Minangkabau architecture",
][:EVAL_PROMPT_COUNT]

PREFERRED_EVAL_IMAGES = {
    "Tongkonan": ["traditional_houses/tongkonan/tongkonan (5).png"],
}


def select_eval_images(images: pd.DataFrame):
    selected = []
    for class_name in EVAL_CLASSES:
        class_images = images[images["class_name"] == class_name].sort_values("image_path").reset_index(drop=True)
        if len(class_images) == 0:
            print(f"Warning: no images found for {class_name}")
            continue

        selected_paths = []
        for preferred_path in PREFERRED_EVAL_IMAGES.get(class_name, []):
            candidate = PROJECT_ROOT / preferred_path
            if candidate.exists():
                selected_paths.append(str(candidate))
            else:
                print(f"Preferred evaluation image not found for {class_name}: {candidate}")

        remaining_needed = max(EVAL_IMAGES_PER_CLASS - len(selected_paths), 0)
        class_images = class_images[~class_images["image_path"].isin(selected_paths)].reset_index(drop=True)
        if remaining_needed > 0:
            if len(class_images) <= remaining_needed:
                indices = list(range(len(class_images)))
            else:
                step = max(len(class_images) // remaining_needed, 1)
                indices = [min(i * step, len(class_images) - 1) for i in range(remaining_needed)]
            selected_paths.extend(class_images.iloc[index]["image_path"] for index in indices)

        for local_index, image_path in enumerate(selected_paths[:EVAL_IMAGES_PER_CLASS], start=1):
            selected.append({
                "class_name": class_name,
                "image_path": str(image_path),
                "image_id": f"{class_name.lower()}_{local_index:02d}",
            })
    return selected


test_images = select_eval_images(images_df)
EVAL_MANIFEST_PATH = EVALUATION_DIR / "eval_manifest.json"
EVAL_MANIFEST_PATH.write_text(json.dumps({
    "run_name": RUN_NAME,
    "seed": SEED,
    "eval_images_per_class": EVAL_IMAGES_PER_CLASS,
    "test_prompts": TEST_PROMPTS,
    "test_images": test_images,
}, indent=2), encoding="utf-8")
for item in test_images:
    print(f"{item['image_id']}: {item['image_path']}")
print(f"Saved evaluation manifest: {EVAL_MANIFEST_PATH}")


def load_img2img_pipeline(path_or_model):
    dtype = torch.float16 if torch.cuda.is_available() else torch.float32
    path = Path(path_or_model)
    if path.exists() and path.suffix.lower() in {".ckpt", ".safetensors"}:
        log_event(f"Loading local Stable Diffusion img2img checkpoint: {path}")
        enable_trusted_legacy_ckpt_loading()
        pipe = StableDiffusionImg2ImgPipeline.from_single_file(
            str(path),
            torch_dtype=dtype,
            safety_checker=None,
        )
    elif path.exists():
        pipe = StableDiffusionImg2ImgPipeline.from_pretrained(
            path,
            torch_dtype=dtype,
            safety_checker=None,
            local_files_only=True,
        )
    else:
        raise FileNotFoundError(f"Local pipeline/checkpoint was not found: {path}")
    if torch.cuda.is_available():
        pipe = pipe.to("cuda")
        pipe.enable_attention_slicing()
    return pipe


def load_dreambooth_checkpoints():
    manifest_path = CHECKPOINT_DIR / "checkpoints_manifest.json"
    if manifest_path.exists():
        checkpoints = json.loads(manifest_path.read_text(encoding="utf-8"))["checkpoints"]
    else:
        checkpoints = [{"step": MAX_TRAIN_STEPS, "label": "final", "pipeline_path": str(CHECKPOINT_DIR / "pipeline")}]
    checkpoints = sorted(checkpoints, key=lambda item: (int(item.get("step", 0)), item.get("label") == "final"))
    return checkpoints


@torch.no_grad()
def run_eval_for_image(image_item: dict, original_pipe, dreambooth_pipe):
    input_image = resize_square(Image.open(image_item["image_path"]), RESOLUTION)
    rows = []
    generator_device = "cuda" if torch.cuda.is_available() else "cpu"
    for prompt_index, prompt in enumerate(TEST_PROMPTS, start=1):
        original_generator = torch.Generator(device=generator_device).manual_seed(SEED + prompt_index)
        dreambooth_generator = torch.Generator(device=generator_device).manual_seed(SEED + prompt_index)
        print(f"  Prompt {prompt_index}/{len(TEST_PROMPTS)}: {prompt}")
        original_image = original_pipe(
            prompt=prompt,
            image=input_image,
            strength=EVAL_STRENGTH,
            num_inference_steps=EVAL_NUM_INFERENCE_STEPS,
            guidance_scale=EVAL_GUIDANCE_SCALE,
            generator=original_generator,
        ).images[0]
        dreambooth_image = dreambooth_pipe(
            prompt=prompt,
            image=input_image,
            strength=EVAL_STRENGTH,
            num_inference_steps=EVAL_NUM_INFERENCE_STEPS,
            guidance_scale=EVAL_GUIDANCE_SCALE,
            generator=dreambooth_generator,
        ).images[0]
        rows.append({
            "prompt_index": prompt_index,
            "prompt": prompt,
            "input": input_image,
            "original": original_image,
            "dreambooth": dreambooth_image,
        })
    return rows


if RUN_EVALUATION:
    assert LOCAL_SD_CKPT_PATH.exists(), f"Stable Diffusion checkpoint not found: {LOCAL_SD_CKPT_PATH}"
    checkpoints_to_evaluate = load_dreambooth_checkpoints()
    log_section("DreamBooth img2img evaluation start")
    log_event(f"Evaluation images: {len(test_images)}; prompts: {len(TEST_PROMPTS)}; checkpoints: {len(checkpoints_to_evaluate)}; strength: {EVAL_STRENGTH}")
    original_pipe = load_img2img_pipeline(LOCAL_SD_CKPT_PATH)
    evaluation_outputs_by_checkpoint = {}
    evaluation_records = []

    for checkpoint in checkpoints_to_evaluate:
        checkpoint_label = checkpoint["label"]
        pipeline_path = Path(checkpoint["pipeline_path"])
        log_section(f"Evaluating checkpoint {checkpoint_label}")
        dreambooth_pipe = load_img2img_pipeline(pipeline_path)
        checkpoint_outputs = {}
        for image_item in test_images:
            print(f"Evaluating {checkpoint_label} | {image_item['image_id']}: {image_item['image_path']}")
            rows = run_eval_for_image(image_item, original_pipe, dreambooth_pipe)
            checkpoint_outputs[image_item["image_id"]] = {
                "checkpoint_label": checkpoint_label,
                "checkpoint_step": checkpoint.get("step"),
                "class_name": image_item["class_name"],
                "image_path": image_item["image_path"],
                "rows": rows,
            }
            for row in rows:
                image_dir = EVALUATION_DIR / "checkpoints" / checkpoint_label / "generated_images" / image_item["image_id"]
                image_dir.mkdir(parents=True, exist_ok=True)
                input_path = image_dir / f"prompt{row['prompt_index']:02d}_input.png"
                original_path = image_dir / f"prompt{row['prompt_index']:02d}_original.png"
                dreambooth_path = image_dir / f"prompt{row['prompt_index']:02d}_dreambooth.png"
                row["input"].save(input_path)
                row["original"].save(original_path)
                row["dreambooth"].save(dreambooth_path)
                for variant, image_path in [("original", original_path), ("dreambooth", dreambooth_path)]:
                    evaluation_records.append({
                        "run_name": RUN_NAME,
                        "checkpoint_label": checkpoint_label,
                        "checkpoint_step": checkpoint.get("step"),
                        "image_id": image_item["image_id"],
                        "class_name": image_item["class_name"],
                        "source_image": image_item["image_path"],
                        "prompt_index": row["prompt_index"],
                        "prompt": row["prompt"],
                        "model_variant": variant,
                        "input_image": str(input_path),
                        "generated_image": str(image_path),
                    })
        evaluation_outputs_by_checkpoint[checkpoint_label] = checkpoint_outputs
        del dreambooth_pipe
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    EVALUATION_RECORDS_PATH = EVALUATION_DIR / "evaluation_records.json"
    EVALUATION_RECORDS_PATH.write_text(json.dumps(evaluation_records, indent=2), encoding="utf-8")
    print(f"Saved evaluation records: {EVALUATION_RECORDS_PATH}")
    del original_pipe
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
else:
    evaluation_records = []
    evaluation_outputs_by_checkpoint = {}
    EVALUATION_RECORDS_PATH = EVALUATION_DIR / "evaluation_records.json"
    EVALUATION_RECORDS_PATH.write_text("[]", encoding="utf-8")
    print("Skipping DreamBooth evaluation.")


## 6. Visualization

In [ ]:
import matplotlib.pyplot as plt


def save_dreambooth_grid(checkpoint_label: str, image_id: str, evaluation_item: dict):
    rows = evaluation_item["rows"]
    n_rows = len(rows)
    fig, axes = plt.subplots(n_rows, 3, figsize=(10.8, 3.65 * n_rows))
    if n_rows == 1:
        axes = [axes]
    column_specs = [("Input image", "input"), ("Original SD img2img", "original"), ("DreamBooth fine-tuned", "dreambooth")]
    for row_idx, row in enumerate(rows):
        for col_idx, (title, key) in enumerate(column_specs):
            ax = axes[row_idx][col_idx]
            ax.imshow(row[key])
            ax.axis("off")
            if row_idx == 0:
                ax.set_title(title, fontsize=12, fontweight="bold")
            if col_idx == 0:
                ax.text(0, -0.12, f"Prompt {row['prompt_index']}: {row['prompt']}", transform=ax.transAxes, fontsize=8.5, ha="left", va="top", wrap=True)
    image_name = Path(evaluation_item["image_path"]).name
    fig.suptitle(f"{RUN_NAME} | {checkpoint_label} | DreamBooth img2img - {image_name}", fontsize=14, fontweight="bold")
    plt.subplots_adjust(wspace=0.04, hspace=0.22, top=0.93, bottom=0.08)
    output_dir = RESULTS_DIR / "checkpoints" / checkpoint_label / "grids"
    output_dir.mkdir(parents=True, exist_ok=True)
    grid_path = output_dir / f"{image_id}_{checkpoint_label}_{RUN_NAME}_dreambooth_img2img_grid.png"
    fig.savefig(grid_path, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"Saved: {grid_path}")


if "evaluation_outputs_by_checkpoint" in globals() and evaluation_outputs_by_checkpoint:
    for checkpoint_label, checkpoint_outputs in evaluation_outputs_by_checkpoint.items():
        for image_id, evaluation_item in checkpoint_outputs.items():
            save_dreambooth_grid(checkpoint_label, image_id, evaluation_item)
else:
    print("No evaluation records to visualize.")

loss_csv = METRICS_DIR / "training_loss.csv"
if loss_csv.exists():
    loss_df = pd.read_csv(loss_csv)
    plt.figure(figsize=(8, 4))
    plt.plot(loss_df["step"], loss_df["loss"], color="#dc2626")
    plt.title(f"DreamBooth Training Loss - {RUN_NAME}")
    plt.xlabel("Optimizer Step")
    plt.ylabel("MSE Loss")
    plt.grid(alpha=0.3)
    loss_plot_path = RESULTS_DIR / f"{RUN_NAME}_training_loss.png"
    plt.savefig(loss_plot_path, dpi=180, bbox_inches="tight")
    plt.show()
    print(f"Saved: {loss_plot_path}")

## 7. Quantitative Metrics

In [ ]:
import numpy as np
from torchvision import transforms as tv_transforms
from torchvision.models import inception_v3, Inception_V3_Weights


def pil_list_from_paths(paths):
    return [resize_square(Image.open(path), RESOLUTION) for path in paths]


def collect_generated_images(records, variant):
    return [Image.open(row["generated_image"]).convert("RGB") for row in records if row["model_variant"] == variant]


def inception_features(images, batch_size=8):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    weights = Inception_V3_Weights.DEFAULT
    model = inception_v3(weights=weights, aux_logits=True)
    model.fc = torch.nn.Identity()
    model.eval().to(device)
    transform = tv_transforms.Compose([
        tv_transforms.Resize((299, 299)),
        tv_transforms.ToTensor(),
        tv_transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    feats = []
    with torch.no_grad():
        for start in range(0, len(images), batch_size):
            batch = torch.stack([transform(img) for img in images[start:start + batch_size]]).to(device)
            feats.append(model(batch).detach().cpu().float())
    return torch.cat(feats, dim=0).numpy()


def frechet_distance(real_features, generated_features, eps=1e-6):
    mu1, mu2 = real_features.mean(axis=0), generated_features.mean(axis=0)
    sigma1 = np.atleast_2d(np.cov(real_features, rowvar=False)) + np.eye(real_features.shape[1]) * eps
    sigma2 = np.atleast_2d(np.cov(generated_features, rowvar=False)) + np.eye(generated_features.shape[1]) * eps
    eigvals = np.linalg.eigvals(sigma1 @ sigma2)
    sqrt_trace = np.sum(np.sqrt(np.clip(eigvals.real, 0, None)))
    diff = mu1 - mu2
    return float(diff @ diff + np.trace(sigma1) + np.trace(sigma2) - 2 * sqrt_trace)


def image_diversity(images):
    if len(images) < 2:
        return 0.0
    arr = np.stack([np.asarray(img.resize((64, 64))).astype(np.float32).reshape(-1) / 255.0 for img in images])
    dists = [float(np.mean(np.abs(arr[i] - arr[j]))) for i in range(len(arr)) for j in range(i + 1, len(arr))]
    return float(np.mean(dists))


def try_clip_scores(records):
    try:
        from transformers import CLIPModel, CLIPProcessor
        device = "cuda" if torch.cuda.is_available() else "cpu"
        model_id = "openai/clip-vit-base-patch32"
        model = CLIPModel.from_pretrained(model_id, local_files_only=True).to(device).eval()
        processor = CLIPProcessor.from_pretrained(model_id, local_files_only=True)
        scores = {"original": [], "dreambooth": []}
        with torch.no_grad():
            for row in records:
                image = Image.open(row["generated_image"]).convert("RGB")
                inputs = processor(text=[row["prompt"]], images=[image], return_tensors="pt", padding=True).to(device)
                outputs = model(**inputs)
                scores[row["model_variant"]].append(torch.nn.functional.cosine_similarity(outputs.image_embeds, outputs.text_embeds).item())
        return {f"clip_text_image_{key}_mean": float(np.mean(values)) if values else None for key, values in scores.items()}
    except Exception as exc:
        print(f"CLIP metric skipped: {exc}")
        return {"clip_text_image_original_mean": None, "clip_text_image_dreambooth_mean": None, "clip_error": str(exc)}

log_section("Metrics computation")
METRICS_PATH = METRICS_DIR / "metrics_summary.json"
if RUN_METRICS and evaluation_records:
    reference_images = pil_list_from_paths(test_df["image_path"].tolist())
    original_images = collect_generated_images(evaluation_records, "original")
    dreambooth_images = collect_generated_images(evaluation_records, "dreambooth")
    real_feats = inception_features(reference_images)
    original_feats = inception_features(original_images)
    dreambooth_feats = inception_features(dreambooth_images)
    metrics_summary = {
        "run_name": RUN_NAME,
        "method": "dreambooth",
        "fid_original_vs_real_gadang": frechet_distance(real_feats, original_feats),
        "fid_dreambooth_vs_real_gadang": frechet_distance(real_feats, dreambooth_feats),
        "diversity_original": image_diversity(original_images),
        "diversity_dreambooth": image_diversity(dreambooth_images),
        "num_reference_images": len(reference_images),
        "num_original_generated_images": len(original_images),
        "num_dreambooth_generated_images": len(dreambooth_images),
    }
    metrics_summary.update(try_clip_scores(evaluation_records))
else:
    metrics_summary = {"run_name": RUN_NAME, "method": "dreambooth", "metrics_skipped": True, "run_metrics": RUN_METRICS, "evaluation_records": len(evaluation_records)}
METRICS_PATH.write_text(json.dumps(metrics_summary, indent=2), encoding="utf-8")
print(json.dumps(metrics_summary, indent=2))
print(f"Saved metrics: {METRICS_PATH}")

## 8. Final Report

In [ ]:
adapter_files = sorted(str(p.relative_to(PROJECT_ROOT)) for p in OUTPUT_DIR.rglob("*") if p.is_file())
print("Exported DreamBooth files:")
for file_name in adapter_files[:80]:
    print("-", file_name)
if len(adapter_files) > 80:
    print(f"... {len(adapter_files) - 80} more files")
print(f"Run config: {RUN_CONFIG_PATH}")
print(f"Split manifest: {SPLIT_MANIFEST_PATH}")
print(f"Evaluation records: {EVALUATION_RECORDS_PATH}")
print(f"Metrics summary: {METRICS_PATH}")
print(f"Root output log: {ROOT_LOG_PATH}")
print(f"Run output log: {RUN_LOG_PATH}")

log_section("Final artifact log")
ARTIFACT_LOG_PATH = OUTPUT_DIR / "artifact_log.json"
artifact_log = {
    "run_name": RUN_NAME,
    "method": RUN_CONFIG.get("method", "lora_concept") if "RUN_CONFIG" in globals() else "unknown",
    "output_dir": str(OUTPUT_DIR),
    "results_dir": str(RESULTS_DIR),
    "run_config": str(RUN_CONFIG_PATH),
    "split_manifest": str(SPLIT_MANIFEST_PATH) if "SPLIT_MANIFEST_PATH" in globals() else None,
    "evaluation_records": str(EVALUATION_RECORDS_PATH) if "EVALUATION_RECORDS_PATH" in globals() else None,
    "metrics_summary": str(METRICS_PATH) if "METRICS_PATH" in globals() else None,
    "root_output_log": str(ROOT_LOG_PATH) if "ROOT_LOG_PATH" in globals() else None,
    "run_output_log": str(RUN_LOG_PATH) if "RUN_LOG_PATH" in globals() else None,
    "files": sorted(str(path.relative_to(OUTPUT_DIR)) for path in OUTPUT_DIR.rglob("*") if path.is_file()),
    "result_files": sorted(str(path.relative_to(RESULTS_DIR)) for path in RESULTS_DIR.rglob("*") if path.is_file()) if RESULTS_DIR.exists() else [],
}
ARTIFACT_LOG_PATH.write_text(json.dumps(artifact_log, indent=2), encoding="utf-8")
print(f"Artifact log: {ARTIFACT_LOG_PATH}")
